## 环境准备：加载 Qwen 大模型

本 Notebook 使用 **ModelScope** 加载 **Qwen2.5-7B-Instruct** 模型，
替代原有的 MockLLM / 模拟 LLM，实现真实的模型推理。

> **低显存备选**：如果 GPU 显存不足，可将模型 ID 替换为 `Qwen/Qwen2.5-3B-Instruct`。

In [ ]:
# ============================================================
# 安装依赖（如需要，取消注释后运行）
# ============================================================
# !pip install modelscope transformers torch -q

# ============================================================
# QwenLLM 封装类：基于 ModelScope 加载 Qwen2.5 模型
# ============================================================
from modelscope import AutoModelForCausalLM, AutoTokenizer
import torch


class QwenLLM:
    """
    基于 ModelScope 的 Qwen2.5 大模型封装类

    支持：
    - system prompt 设置
    - 多轮对话上下文维护
    - GPU / CPU 自动检测
    - 温度与生成长度控制
    """

    def __init__(self, model_name="Qwen/Qwen2.5-7B-Instruct", device=None):
        """
        初始化 Qwen 模型

        Args:
            model_name: 模型 ID，默认 7B；低显存可改为 "Qwen/Qwen2.5-3B-Instruct"
            device: 指定设备，None 表示自动检测
        """
        # GPU / CPU 自动检测
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        print(f"[QwenLLM] 使用设备: {self.device}")
        print(f"[QwenLLM] 加载模型: {model_name}")
        print(f"[QwenLLM] 提示: 如显存不足，可替换为 Qwen/Qwen2.5-3B-Instruct")

        # 加载模型和分词器
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # 多轮对话历史
        self.messages = []

        print(f"[QwenLLM] 模型加载完成")

    def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
        """
        对话接口

        Args:
            user_message: 用户消息
            system_prompt: 系统提示词（可选）
            max_new_tokens: 最大生成 token 数
            temperature: 采样温度

        Returns:
            模型生成的回复文本
        """
        # 构建消息列表
        if system_prompt:
            messages = [{"role": "system", "content": system_prompt}]
        else:
            messages = []
        messages.extend(self.messages)
        messages.append({"role": "user", "content": user_message})

        # 应用聊天模板
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        # 生成回复
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True
        )
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # 更新对话历史
        self.messages.append({"role": "user", "content": user_message})
        self.messages.append({"role": "assistant", "content": response})

        return response

    def reset(self):
        """清空对话历史"""
        self.messages = []


# 初始化模型（首次运行需要下载，请耐心等待）
llm = QwenLLM(model_name="Qwen/Qwen2.5-7B-Instruct")
# 如显存不足，请使用：llm = QwenLLM(model_name="Qwen/Qwen2.5-3B-Instruct")

print("\n模型就绪，可以在后续 cell 中使用 llm.chat() 进行对话")

# 02 - 工具调用与 Function Calling：扩展 Agent 的能力边界

## 学习目标

- 理解 Function Calling 的原理与机制
- 掌握 OpenAI / 国产大模型的 Function Calling API
- 学习如何设计和注册 Agent 工具
- 实现一个支持多工具调用的 Agent

---

## 1. Function Calling 概述

### 1.1 什么是 Function Calling？

**Function Calling**（函数调用）是 LLM 的一项核心能力，允许模型：

1. **识别**何时需要调用外部工具
2. **生成**符合工具签名的参数
3. **接收**工具执行结果并继续推理

**核心流程**：

```
用户提问 → LLM 分析 → 判断需要工具 → 生成工具调用 →
执行工具 → 返回结果 → LLM 整合 → 生成最终回答
```

### 1.2 为什么需要 Function Calling？

| 能力 | 说明 | 示例 |
|------|------|------|
| **获取实时信息** | LLM 知识有截止日期 | 查询今天天气、股价 |
| **执行计算** | 避免 LLM 数学错误 | 复杂数学运算 |
| **操作外部系统** | 与数据库/API 交互 | 预订酒店、发送邮件 |
| **访问专有数据** | 企业私有知识 | 查询内部文档、客户信息 |
| **执行代码** | 运行程序验证结果 | 数据分析、代码测试 |

---

## 2. Function Calling 的核心机制

### 2.1 工具定义格式（OpenAI 风格）

```json
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "获取指定城市的当前天气",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "城市名称，如北京、上海"
        },
        "unit": {
          "type": "string",
          "enum": ["celsius", "fahrenheit"],
          "description": "温度单位"
        }
      },
      "required": ["city"]
    }
  }
}
```

### 2.2 调用流程

```
Step 1: 定义工具
    └── 名称、描述、参数schema

Step 2: 发送请求
    └── messages + tools

Step 3: 模型判断
    ├── 不需要工具 → 直接回答
    └── 需要工具 → 返回 tool_calls

Step 4: 执行工具
    └── 解析参数 → 调用函数 → 获取结果

Step 5: 发送结果
    └── 将结果加入 messages → 再次请求

Step 6: 生成回答
    └── 模型基于工具结果生成最终回答
```

---

## 3. 动手实践：Function Calling 完整示例

### 3.1 基础设置

In [ ]:
# ============================================================
# LLM 接口定义
# ============================================================
#
# 重要说明：Function Calling 需要模型原生支持工具调用格式。
# 纯 transformers 推理（如 QwenLLM）不支持原生 Function Calling，
# 因此本文件保留 MockLLM 实现，用于演示 Function Calling 流程。
#
# 如需真实 Function Calling，建议使用以下 API 服务：
#   - OpenAI API (gpt-4 / gpt-3.5-turbo)
#   - 通义千问 API (dashscope)
#   - 智谱 AI API (zhipuai)
#   - DeepSeek API
#
# 以下 MockLLM 已标注为「无模型时的备选方案」。
# ============================================================

import json
import os
from typing import List, Dict, Callable, Any
from datetime import datetime

# ---- 无模型时的备选方案：MockLLM（用于演示 Function Calling 流程）----
class MockLLM:
    """
    模拟支持 Function Calling 的 LLM

    说明：纯 transformers 推理不支持原生 Function Calling，
    此 MockLLM 仅用于演示工具调用的流程和接口设计。
    生产环境请使用 OpenAI / 通义千问 / 智谱 AI 等支持 FC 的 API。
    """

    def __init__(self):
        self.available_tools = {}

    def register_tools(self, tools: List[Dict]):
        """注册可用工具"""
        for tool in tools:
            self.available_tools[tool["function"]["name"]] = tool

    def chat_completion(self, messages: List[Dict], tools: List[Dict] = None) -> Dict:
        """
        模拟聊天完成 API

        实际应用中替换为：
        - OpenAI: openai.ChatCompletion.create()
        - 通义千问: dashscope.Generation.call()
        - 智谱 AI: zhipuai.ZhipuAI().chat.completions.create()
        """

        last_message = messages[-1]["content"]

        # 模拟模型判断是否需要调用工具
        if tools:
            for tool in tools:
                tool_name = tool["function"]["name"]
                tool_desc = tool["function"]["description"]

                # 简单关键词匹配模拟工具选择
                if self._should_use_tool(last_message, tool_name, tool_desc):
                    params = self._extract_params(last_message, tool)
                    return {
                        "role": "assistant",
                        "content": None,
                        "tool_calls": [{
                            "id": f"call_{tool_name}_001",
                            "type": "function",
                            "function": {
                                "name": tool_name,
                                "arguments": json.dumps(params)
                            }
                        }]
                    }

        # 直接回答（使用 QwenLLM 生成）
        response = llm.chat(
            last_message,
            system_prompt="你是一个智能助手，可以使用工具来帮助用户。请根据需要使用可用工具。"
        )
        return {
            "role": "assistant",
            "content": response
        }

    def _should_use_tool(self, message: str, tool_name: str, tool_desc: str) -> bool:
        """判断是否应该使用工具"""
        keywords = {
            "get_weather": ["天气", "温度", "下雨", "晴天"],
            "calculate": ["计算", "等于", "多少"],
            "search_web": ["搜索", "查询", "查找", "什么是"],
            "get_datetime": ["时间", "日期", "现在", "今天"],
        }

        tool_keywords = keywords.get(tool_name, [])
        return any(kw in message for kw in tool_keywords)

    def _extract_params(self, message: str, tool: Dict) -> Dict:
        """从消息中提取参数"""
        params = {}
        properties = tool["function"]["parameters"]["properties"]

        for param_name, param_info in properties.items():
            if param_name == "city":
                cities = ["北京", "上海", "广州", "深圳", "杭州"]
                for city in cities:
                    if city in message:
                        params[param_name] = city
                        break
                if param_name not in params:
                    params[param_name] = "北京"
            elif param_name == "expression":
                import re
                match = re.search(r'(\d+\s*[+\-*/]\s*\d+)', message)
                params[param_name] = match.group(1) if match else "1+1"
            elif param_name == "query":
                params[param_name] = message
            elif param_name == "unit":
                params[param_name] = "celsius"
            else:
                params[param_name] = ""

        return params

# ---- 备选方案结束 ----

print("MockLLM 初始化完成（Function Calling 演示用）")
print("提示：工具调用判断使用关键词匹配，最终回答使用 QwenLLM 生成")

### 3.2 定义工具集

In [ ]:
# 定义工具函数

def get_weather(city: str, unit: str = "celsius") -> str:
    """
    获取指定城市的天气

    Args:
        city: 城市名称
        unit: 温度单位 (celsius/fahrenheit)
    """
    weather_db = {
        "北京": {"temp": 25, "condition": "晴天", "humidity": "45%"},
        "上海": {"temp": 28, "condition": "多云", "humidity": "60%"},
        "广州": {"temp": 32, "condition": "小雨", "humidity": "75%"},
        "深圳": {"temp": 31, "condition": "阴天", "humidity": "70%"},
        "杭州": {"temp": 27, "condition": "晴转多云", "humidity": "55%"}
    }

    data = weather_db.get(city, {"temp": 20, "condition": "未知", "humidity": "50%"})

    temp = data["temp"]
    if unit == "fahrenheit":
        temp = temp * 9/5 + 32
        unit_symbol = "°F"
    else:
        unit_symbol = "°C"

    return f"{city}当前天气：{data['condition']}，温度 {temp:.1f}{unit_symbol}，湿度 {data['humidity']}"

def calculate(expression: str) -> str:
    """
    执行数学计算

    Args:
        expression: 数学表达式，如 "15 * 23"
    """
    try:
        # 安全计算：只允许基本运算符
        allowed_chars = set('0123456789+-*/.() ')
        if not all(c in allowed_chars for c in expression):
            return "错误：表达式包含非法字符"

        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"计算错误: {str(e)}"

def search_web(query: str) -> str:
    """
    模拟网络搜索

    Args:
        query: 搜索关键词
    """
    # 模拟搜索结果
    search_db = {
        "Python": "Python 是一种高级编程语言，由 Guido van Rossum 于 1991 年创建。",
        "AI Agent": "AI Agent 是一种能够自主感知、推理和行动的智能系统。",
        "大模型": "大语言模型（LLM）是基于 Transformer 架构的深度学习模型。"
    }

    for key, value in search_db.items():
        if key in query:
            return f"搜索结果：{value}"

    return f"搜索 '{query}' 完成，找到 3 条相关结果。"

def get_datetime() -> str:
    """获取当前日期时间"""
    now = datetime.now()
    return f"当前时间：{now.strftime('%Y年%m月%d日 %H:%M:%S')}"

# 工具注册表
tools_registry = {
    "get_weather": get_weather,
    "calculate": calculate,
    "search_web": search_web,
    "get_datetime": get_datetime
}

print("✅ 工具函数定义完成")
print(f"已注册 {len(tools_registry)} 个工具")
for name in tools_registry:
    print(f"  - {name}")

### 3.3 定义工具 Schema

In [ ]:
# 定义工具 Schema（OpenAI 格式）
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取指定城市的当前天气信息，包括温度、天气状况和湿度",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称，如北京、上海、广州、深圳"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "温度单位，默认为摄氏度"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "执行数学计算，支持加减乘除和括号运算",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "数学表达式，如 '15 * 23'、'(10 + 5) / 3'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "搜索互联网获取信息，适用于查询实时信息或知识性问题",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "搜索关键词或问题"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_datetime",
            "description": "获取当前的日期和时间",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    }
]

print("✅ 工具 Schema 定义完成")
print(json.dumps(tools_schema, indent=2, ensure_ascii=False))

### 3.4 构建 Function Calling Agent

In [ ]:
class FunctionCallingAgent:
    """
    基于 Function Calling 的 Agent

    支持多轮工具调用，直到任务完成
    """

    def __init__(self, llm, tools_schema: List[Dict], tools_registry: Dict[str, Callable]):
        self.llm = llm
        self.tools_schema = tools_schema
        self.tools_registry = tools_registry
        self.max_iterations = 10

        # 注册工具到 LLM
        self.llm.register_tools(tools_schema)

    def run(self, user_query: str) -> str:
        """
        运行 Agent

        Args:
            user_query: 用户查询

        Returns:
            最终回答
        """
        print(f"\n{'='*60}")
        print(f"🚀 用户查询: {user_query}")
        print(f"{'='*60}")

        # 初始化消息历史
        messages = [
            {
                "role": "system",
                "content": "你是一个智能助手，可以使用工具来帮助用户。请根据需要使用可用工具。"
            },
            {
                "role": "user",
                "content": user_query
            }
        ]

        for i in range(self.max_iterations):
            print(f"\n📍 第 {i+1} 轮交互")
            print("-" * 40)

            # 调用 LLM
            response = self.llm.chat_completion(messages, self.tools_schema)

            # 检查是否需要调用工具
            if response.get("tool_calls"):
                # 处理工具调用
                for tool_call in response["tool_calls"]:
                    tool_name = tool_call["function"]["name"]
                    tool_args = json.loads(tool_call["function"]["arguments"])

                    print(f"🔧 调用工具: {tool_name}")
                    print(f"   参数: {tool_args}")

                    # 执行工具
                    if tool_name in self.tools_registry:
                        try:
                            result = self.tools_registry[tool_name](**tool_args)
                            print(f"   结果: {result}")
                        except Exception as e:
                            result = f"工具执行错误: {str(e)}"
                            print(f"   错误: {result}")
                    else:
                        result = f"未知工具: {tool_name}"
                        print(f"   错误: {result}")

                    # 将工具结果添加到消息历史
                    messages.append({
                        "role": "assistant",
                        "content": None,
                        "tool_calls": [tool_call]
                    })
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call["id"],
                        "content": str(result)
                    })
            else:
                # 直接返回回答
                answer = response["content"]
                print(f"\n✅ 最终回答: {answer}")
                return answer

        print("\n⚠️ 达到最大迭代次数")
        return "处理超时，请重试"

# 创建 Agent（Function Calling 场景保留 MockLLM，最终回答使用 QwenLLM）
llm_mock = MockLLM()
agent = FunctionCallingAgent(llm_mock, tools_schema, tools_registry)

print("✅ Function Calling Agent 初始化完成")

In [ ]:
# 测试 1: 天气查询
result = agent.run("今天上海的天气怎么样？")

In [ ]:
# 测试 2: 数学计算
result = agent.run("帮我计算 125 除以 5 再加上 30 等于多少？")

In [ ]:
# 测试 3: 获取时间
result = agent.run("现在几点了？")

---

## 4. 工具设计的最佳实践

### 4.1 工具命名规范

```python
# ✅ 好的命名
get_weather        # 动词 + 名词，清晰表达功能
search_documents     # 搜索文档
send_email           # 发送邮件
create_calendar_event # 创建日历事件

# ❌ 不好的命名
weather              # 缺少动词
func1                # 无意义名称
do_something         # 过于模糊
```

### 4.2 描述编写要点

```python
# ✅ 好的描述
"""
获取指定城市的当前天气信息，包括温度、天气状况和湿度。
适用于用户询问天气相关问题时调用。
"""

# ❌ 不好的描述
"""天气工具"""  # 过于简单，模型无法理解何时使用
```

### 4.3 参数设计原则

| 原则 | 说明 | 示例 |
|------|------|------|
| **明确类型** | 指定参数的数据类型 | `"type": "string"` |
| **添加描述** | 说明参数的用途和格式 | `"description": "城市名称"` |
| **设置枚举** | 限制可选值 | `"enum": ["celsius", "fahrenheit"]` |
| **标记必填** | 明确哪些参数必须提供 | `"required": ["city"]` |
| **提供示例** | 帮助模型理解格式 | 在描述中包含示例 |

### 4.4 错误处理

```python
def robust_tool(func):
    """工具装饰器：添加错误处理和日志"""
    def wrapper(*args, **kwargs):
        try:
            print(f"[工具调用] {func.__name__}({kwargs})")
            result = func(*args, **kwargs)
            print(f"[工具结果] {result}")
            return result
        except Exception as e:
            error_msg = f"工具 {func.__name__} 执行失败: {str(e)}"
            print(f"[工具错误] {error_msg}")
            return error_msg
    return wrapper

@robust_tool
def get_weather(city: str):
    # 工具实现
    pass
```

---

## 5. 多工具调用场景

### 5.1 串行调用

一个工具的输出作为下一个工具的输入。

```
用户："帮我查一下北京明天的天气，如果下雨就提醒我带伞"

Step 1: get_weather(city="北京", date="明天")
    → 结果：明天有小雨

Step 2: send_reminder(message="明天北京有小雨，记得带伞！")
    → 结果：提醒已设置
```

### 5.2 并行调用

多个工具同时调用，结果合并。

```
用户："比较一下北京和上海今天的天气"

Step 1: get_weather(city="北京")  [并行]
Step 2: get_weather(city="上海")  [并行]

Step 3: 合并结果并生成对比回答
```

### 5.3 条件调用

根据前面工具的结果决定后续操作。

```
用户："查询订单 12345 的状态，如果已发货就查询物流"

Step 1: get_order_status(order_id="12345")
    → 结果：状态 = "已发货"

Step 2: 判断：状态 == "已发货" → 查询物流

Step 3: get_tracking_info(order_id="12345")
    → 结果：物流信息
```

---

## 6. 小结

### 核心要点

1. **Function Calling** 让 LLM 能够识别何时需要外部工具并生成正确参数
2. **核心流程**：定义工具 → 发送请求 → 模型判断 → 执行工具 → 返回结果 → 生成回答
3. **工具设计**：命名清晰、描述详细、参数明确、错误处理完善
4. **调用模式**：串行、并行、条件调用，适应不同场景

### 下一步

- [03_memory_systems.ipynb](03_memory_systems.ipynb) - 学习 Agent 记忆系统设计
- [04_planning_and_reflection.ipynb](04_planning_and_reflection.ipynb) - 掌握规划与反思模式

---

## 参考资源

- [OpenAI Function Calling 文档](https://platform.openai.com/docs/guides/function-calling)
- [LangChain Tools 文档](https://python.langchain.com/docs/modules/agents/tools/)
- [Toolformer: Language Models Can Teach Themselves to Use Tools](https://arxiv.org/abs/2302.04761)